# 01 — Build the two core tables
Tests `roche_poc.data`. Input: raw CSVs in `data/raw/Snapshot_2026/`. Output: `data/processed/snapshots.parquet` and `comments.parquet`.

**Rule:** logic lives in `src/roche_poc/data/`; this notebook only calls it and inspects the result.

In [ ]:
# Works after `pip install -e .`; the fallback below covers a fresh clone.
import sys, pathlib
_src = pathlib.Path.cwd().resolve().parent / "src"
if _src.exists() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))
import pandas as pd
pd.set_option("display.max_columns", 30, "display.width", 160)

In [ ]:
from roche_poc.data.build_tables import build_all
report = build_all()
report

## Sanity checks
Every cleaning step reports what it touched (no silent drops).

In [ ]:
from roche_poc import config
from roche_poc.data.build_tables import load_table
snapshots = load_table(config.SNAPSHOTS_PATH)
comments = load_table(config.COMMENTS_PATH)
print('snapshots', snapshots.shape, '| comments', comments.shape)
snapshots.groupby(config.COL_SNAPSHOT).size().rename('rows per snapshot')

In [ ]:
# Share of rows with a comment / a root cause, per status: shows the governance gap on critical rows
snapshots.assign(has_comment=snapshots[config.COL_COMMENT].notna(), has_root_cause=snapshots[config.COL_ROOT_CAUSE].notna()).groupby(config.COL_STATUS)[['has_comment','has_root_cause']].mean().round(3)

In [ ]:
snapshots[config.COL_STATUS].value_counts(dropna=False), snapshots[config.COL_IMPACT].value_counts(dropna=False)